<a href="https://colab.research.google.com/github/e23323-dot/Statistical-Learning-e23323/blob/main/Assignment04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install plotly scikit-learn scipy pandas numpy

In [4]:
from google.colab import files
import pandas as pd
import numpy as np
import io
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
import warnings
warnings.filterwarnings('ignore')


class PlottingMethods:

    @staticmethod
    def create_bar_chart(data, x, y=None, title="Bar Chart", xlabel=None, ylabel=None):
        fig = px.bar(data, x=x, y=y, title=title, labels={x: xlabel or x, y: ylabel or y})
        return fig.to_html(full_html=False)

    @staticmethod
    def create_pie_chart(data, names, values, title="Pie Chart"):
        fig = px.pie(data, names=names, values=values, title=title)
        return fig.to_html(full_html=False)

    @staticmethod
    def create_histogram(data, column, bins=30, title="Histogram", xlabel=None):
        fig = px.histogram(data, x=column, nbins=bins, title=title,
                          labels={column: xlabel or column})
        return fig.to_html(full_html=False)


class DataInspector:

    def __init__(self):
        self.data = None
        self.numeric_cols = []
        self.categorical_cols = []
        self.original_data = None

    def upload_data(self):
        print("Please upload a CSV file:")
        uploaded = files.upload()

        for filename in uploaded.keys():
            self.data = pd.read_csv(io.BytesIO(uploaded[filename]))
            self.original_data = self.data.copy()
            print(f"Successfully loaded: {filename}")
            print(f"Shape: {self.data.shape}")
            self._identify_column_types()
            return self.data

    def _identify_column_types(self):
        self.numeric_cols = self.data.select_dtypes(include=[np.number]).columns.tolist()
        self.categorical_cols = self.data.select_dtypes(include=['object', 'category']).columns.tolist()

    def _convert_garbage_to_nan(self):
        garbage_values = ['?', 'n/a', 'NULL', '', 'None', 'null', 'NaN', 'nan']
        self.data = self.data.replace(garbage_values, np.nan)

    def _auto_type_correction(self):
        for col in self.data.columns:
            if self.data[col].dtype == 'object':
                try:
                    converted = pd.to_numeric(self.data[col])
                    if not converted.isna().all():
                        self.data[col] = converted
                        print(f"Converted '{col}' to numeric")
                except:
                    pass
        self._identify_column_types()

    def data_summary(self):
        print("\n" + "="*60)
        print("DATA SUMMARY REPORT")
        print("="*60)
        print(f"Rows: {self.data.shape[0]:,} | Columns: {self.data.shape[1]:,}")
        print(f"Numeric columns: {len(self.numeric_cols)}")
        print(f"Categorical columns: {len(self.categorical_cols)}")

        print("\nFIRST 20 ROWS:")
        print(self.data.head(20))

        print("\nNUMERIC COLUMNS:")
        print(self.data[self.numeric_cols].describe() if self.numeric_cols else "None")

        print("\nCATEGORICAL COLUMNS:")
        for col in self.categorical_cols:
            print(f"  • {col}: {self.data[col].nunique()} unique values")

    def handle_missing_values(self, strategy='mean', constant=None):
        print(f"\nHandling missing values with strategy: {strategy}")

        self._convert_garbage_to_nan()
        self._auto_type_correction()

        for col in self.data.columns:
            if self.data[col].isna().sum() > 0:
                if col in self.numeric_cols:
                    if strategy == 'mean':
                        self.data[col].fillna(self.data[col].mean(), inplace=True)
                    elif strategy == 'median':
                        self.data[col].fillna(self.data[col].median(), inplace=True)
                    elif strategy == 'mode':
                        self.data[col].fillna(self.data[col].mode()[0] if not self.data[col].mode().empty else 0, inplace=True)
                    elif strategy == 'constant' and constant is not None:
                        self.data[col].fillna(constant, inplace=True)
                else:
                    if strategy == 'mode':
                        self.data[col].fillna(self.data[col].mode()[0] if not self.data[col].mode().empty else 'Unknown', inplace=True)
                    elif strategy == 'constant' and constant is not None:
                        self.data[col].fillna(constant, inplace=True)

        print(f"Missing values handled. Remaining nulls: {self.data.isna().sum().sum()}")
        return self.data

    def remove_duplicates(self):
        initial = len(self.data)
        self.data.drop_duplicates(inplace=True)
        removed = initial - len(self.data)
        print(f"Removed {removed} duplicate rows")
        return self.data

    def handle_outliers(self, columns=None, method='flag', iqr_multiplier=1.5):
        if columns is None:
            columns = self.numeric_cols

        outliers_removed = 0
        outlier_flags = pd.Series([False] * len(self.data), name='is_outlier')

        for col in columns:
            if col in self.numeric_cols:
                Q1 = self.data[col].quantile(0.25)
                Q3 = self.data[col].quantile(0.75)
                IQR = Q3 - Q1
                lower = Q1 - iqr_multiplier * IQR
                upper = Q3 + iqr_multiplier * IQR

                col_outliers = (self.data[col] < lower) | (self.data[col] > upper)
                outlier_flags |= col_outliers

                if method == 'delete':
                    outliers_removed += col_outliers.sum()

        if method == 'flag':
            self.data['is_outlier'] = outlier_flags
            print(f"Flagged {outlier_flags.sum()} rows as outliers")
        elif method == 'delete':
            self.data = self.data[~outlier_flags]
            print(f"Removed {outliers_removed} outlier rows")

        return self.data

    def delete_rows(self, row_indices_input):
        indices = [int(i.strip()) for i in row_indices_input.split(',')]
        initial = len(self.data)
        self.data = self.data.drop(indices, errors='ignore')
        print(f"Removed {initial - len(self.data)} rows")
        return self.data

    def delete_columns(self, column_names_input):
        columns = [col.strip() for col in column_names_input.split(',')]
        existing_cols = [col for col in columns if col in self.data.columns]
        self.data = self.data.drop(columns=existing_cols)
        print(f"Removed columns: {existing_cols}")
        self._identify_column_types()
        return self.data

    def extract_normalized_numeric_data(self, scaling='minmax', columns=None):
        if columns is None:
            columns = self.numeric_cols

        if not columns:
            print("No numeric columns to scale")
            return pd.DataFrame()

        scalers = {
            'minmax': MinMaxScaler(),
            'standard': StandardScaler(),
            'robust': RobustScaler()
        }

        scaler = scalers.get(scaling)
        if not scaler:
            raise ValueError(f"Invalid scaling method. Choose from {list(scalers.keys())}")

        scaled_data = scaler.fit_transform(self.data[columns])
        scaled_df = pd.DataFrame(scaled_data, columns=[f"{col}_scaled" for col in columns], index=self.data.index)

        print(f"Applied {scaling} scaling to {len(columns)} columns")
        return scaled_df

    def extract_normalized_categorical_data(self, encoding='onehot', columns=None):
        if columns is None:
            columns = self.categorical_cols

        if not columns:
            print("No categorical columns to encode")
            return pd.DataFrame()

        if encoding == 'onehot':
            encoded = pd.get_dummies(self.data[columns], prefix=columns)
            print(f"One-hot encoded {len(columns)} columns -> {encoded.shape[1]} features")

        elif encoding == 'ordinal':
            encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
            encoded_array = encoder.fit_transform(self.data[columns].astype(str))
            encoded = pd.DataFrame(encoded_array, columns=columns, index=self.data.index)
            print(f"Ordinal encoded {len(columns)} columns")

        elif encoding == 'uniform':
            for col in columns:
                unique_vals = self.data[col].nunique()
                if unique_vals > 1:
                    encoder = OrdinalEncoder()
                    encoded_vals = encoder.fit_transform(self.data[col].fillna('Unknown').astype(str).values.reshape(-1, 1))
                    normalized = (encoded_vals - encoded_vals.min()) / (encoded_vals.max() - encoded_vals.min())
                    self.data[f"{col}_uniform"] = normalized
            encoded = self.data[[f"{col}_uniform" for col in columns if f"{col}_uniform" in self.data.columns]]
            print(f"Uniform encoded {len(columns)} columns")
        else:
            raise ValueError(f"Invalid encoding method. Choose from 'onehot', 'ordinal', 'uniform'")

        return encoded

    def merge_normalized_data(self, numeric_df=None, categorical_df=None):
        result = self.data.copy()

        if numeric_df is not None:
            result = pd.concat([result, numeric_df], axis=1)
        if categorical_df is not None:
            result = pd.concat([result, categorical_df], axis=1)

        print(f"Merged dataset shape: {result.shape}")
        return result

    def plot_univariate_subplots(self, columns=None):
        if columns is None:
            columns = self.numeric_cols[:3]

        for col in columns:
            fig = make_subplots(rows=1, cols=3,
                               subplot_titles=[f"{col} - Distribution", f"{col} - Scatter", f"{col} - Histogram"])

            fig.add_trace(go.Violin(y=self.data[col], name=col, box_visible=True, line_color='black'), row=1, col=1)
            fig.add_trace(go.Scatter(x=self.data.index, y=self.data[col], mode='markers', marker=dict(size=4, opacity=0.6)), row=1, col=2)
            fig.add_trace(go.Histogram(x=self.data[col], nbinsx=30), row=1, col=3)

            fig.update_layout(height=500, title_text=f"Univariate Analysis: {col}", showlegend=False)
            fig.show()

    def plot_relationship(self, col1, col2):
        type1 = 'numeric' if col1 in self.numeric_cols else 'categorical'
        type2 = 'numeric' if col2 in self.numeric_cols else 'categorical'

        if type1 == 'numeric' and type2 == 'numeric':
            fig = px.scatter(self.data, x=col1, y=col2, trendline='ols',
                            title=f"{col1} vs {col2} (Correlation: {self.data[col1].corr(self.data[col2]):.3f})")
            fig.show()

        elif (type1 == 'numeric' and type2 == 'categorical') or (type1 == 'categorical' and type2 == 'numeric'):
            num_col = col1 if type1 == 'numeric' else col2
            cat_col = col2 if type1 == 'numeric' else col1
            fig = px.box(self.data, x=cat_col, y=num_col, points='all',
                        title=f"{num_col} by {cat_col}")
            fig.show()

        else:
            contingency = pd.crosstab(self.data[col1], self.data[col2])
            fig = px.bar(contingency, barmode='group', title=f"{col1} vs {col2}")
            fig.show()

    def plot_categorical_frequency(self, column, top_n=10):
        freq = self.data[column].value_counts().head(top_n)
        pct = (freq / len(self.data) * 100).round(1)

        fig = go.Figure(data=[
            go.Bar(x=freq.index.astype(str), y=freq.values,
                  text=[f"{v}<br>({p}%)" for v, p in zip(freq.values, pct)],
                  textposition='outside')
        ])
        fig.update_layout(title=f"Frequency: {column}", xaxis_title=column, yaxis_title="Count")
        fig.show()

    def plot_all_associations_heatmap(self):
        all_cols = self.numeric_cols + self.categorical_cols
        n = len(all_cols)
        association_matrix = np.zeros((n, n))

        for i, col1 in enumerate(all_cols):
            for j, col2 in enumerate(all_cols):
                if i == j:
                    association_matrix[i, j] = 1.0
                elif i > j:
                    association_matrix[i, j] = association_matrix[j, i]
                else:
                    if col1 in self.numeric_cols and col2 in self.numeric_cols:
                        association_matrix[i, j] = self.data[col1].corr(self.data[col2])
                    elif col1 in self.categorical_cols and col2 in self.categorical_cols:
                        contingency = pd.crosstab(self.data[col1], self.data[col2])
                        chi2 = stats.chi2_contingency(contingency)[0]
                        n_total = contingency.sum().sum()
                        min_dim = min(contingency.shape) - 1
                        association_matrix[i, j] = np.sqrt(chi2 / (n_total * min_dim)) if min_dim > 0 else 0
                    else:
                        num_col = col1 if col1 in self.numeric_cols else col2
                        cat_col = col2 if col1 in self.numeric_cols else col1
                        encoded = pd.get_dummies(self.data[cat_col], drop_first=True)
                        if encoded.shape[1] > 0:
                            correlation = self.data[num_col].corr(encoded.iloc[:, 0])
                        else:
                            correlation = 0
                        association_matrix[i, j] = abs(correlation)

        fig = px.imshow(association_matrix, x=all_cols, y=all_cols,
                       color_continuous_scale='RdBu_r', text_auto='.2f',
                       title="Association Heatmap")
        fig.update_layout(height=max(600, 30 * n), width=max(600, 30 * n))
        fig.show()


if __name__ == "__main__":
    print("="*60)
    print("DATA INSPECTOR - DEMONSTRATION")
    print("="*60)

    inspector = DataInspector()

    print("\nLoading Titanic dataset...")
    titanic_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    inspector.data = pd.read_csv(titanic_url)
    inspector.original_data = inspector.data.copy()
    inspector._identify_column_types()

    inspector.data_summary()
    inspector.handle_missing_values(strategy='median')
    inspector.remove_duplicates()
    inspector.handle_outliers(columns=['Age', 'Fare'], method='flag')

    numeric_scaled = inspector.extract_normalized_numeric_data(scaling='minmax', columns=['Age', 'Fare'])
    categorical_encoded = inspector.extract_normalized_categorical_data(encoding='onehot', columns=['Sex', 'Embarked'])

    merged = inspector.merge_normalized_data(numeric_scaled, categorical_encoded)

    inspector.plot_univariate_subplots(['Age', 'Fare'])
    inspector.plot_relationship('Age', 'Fare')
    inspector.plot_relationship('Sex', 'Survived')
    inspector.plot_categorical_frequency('Pclass')
    inspector.plot_all_associations_heatmap()

    plotter = PlottingMethods()
    print("\nHTML output example ready")

    print(f"\nFinal dataset shape: {inspector.data.shape}")

DATA INSPECTOR - DEMONSTRATION

Loading Titanic dataset...

DATA SUMMARY REPORT
Rows: 891 | Columns: 12
Numeric columns: 7
Categorical columns: 5

FIRST 20 ROWS:
    PassengerId  Survived  Pclass  \
0             1         0       3   
1             2         1       1   
2             3         1       3   
3             4         1       1   
4             5         0       3   
5             6         0       3   
6             7         0       1   
7             8         0       3   
8             9         1       3   
9            10         1       2   
10           11         1       3   
11           12         1       1   
12           13         0       3   
13           14         0       3   
14           15         0       3   
15           16         1       2   
16           17         0       3   
17           18         1       2   
18           19         0       3   
19           20         1       3   

                                                 Name     Se


HTML output example ready

Final dataset shape: (891, 13)
